In [ ]:
# ── Reproducibility Header ────────────────────────────────────────────

import sys, random
import numpy as np
import warnings

RANDOM_SEED = 414  # Course constant. Do not change.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

# Environment check
print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Seed    : {RANDOM_SEED}')


Python  : 3.13.7
NumPy   : 2.4.2
Seed    : 414


In [ ]:
# ──  FastF1 Session Load (Bahrain 2024 Race) ─────────────────────────
import fastf1
import os

# Import cache of f1 if exists for faster load of data
os.makedirs('fastf1_cache', exist_ok=True)
fastf1.Cache.enable_cache('fastf1_cache')


# Load one complete session from the 2024 season
# The race will be 'Bahrain'
session = fastf1.get_session(2024, 'Bahrain', 'R')
session.load(laps=True, telemetry=False, weather=False, messages=False)

# Required outputs
print(f"Session name: {session.name}")
print(f"Event name  : {session.event['EventName']}")
print(f"Results shape (rows, cols): {session.results.shape}")
print(f"Results columns: {list(session.results.columns)}")

print('\nFirst 5 rows of lap data (session.laps.head()):')
display(session.laps.head(5))

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']


Session name: Race
Event name  : Bahrain Grand Prix
Results shape (rows, cols): (20, 22)
Results columns: ['DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName', 'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName', 'HeadshotUrl', 'CountryCode', 'Position', 'ClassifiedPosition', 'GridPosition', 'Q1', 'Q2', 'Q3', 'Time', 'Status', 'Points', 'Laps']

First 5 rows of lap data (session.laps.head()):


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,0 days 01:01:37.489000,VER,1,0 days 00:01:37.284000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:41.266000,...,False,Red Bull Racing,0 days 00:59:59.911000,NaT,12,1.0,None,,False,False
1,0 days 01:03:13.785000,VER,1,0 days 00:01:36.296000,2.0,1.0,NaT,NaT,0 days 00:00:30.916000,0 days 00:00:41.661000,...,False,Red Bull Racing,0 days 01:01:37.489000,NaT,1,1.0,None,,False,True
2,0 days 01:04:50.538000,VER,1,0 days 00:01:36.753000,3.0,1.0,NaT,NaT,0 days 00:00:30.999000,0 days 00:00:41.966000,...,False,Red Bull Racing,0 days 01:03:13.785000,NaT,1,1.0,None,,False,True
3,0 days 01:06:27.185000,VER,1,0 days 00:01:36.647000,4.0,1.0,NaT,NaT,0 days 00:00:30.931000,0 days 00:00:41.892000,...,False,Red Bull Racing,0 days 01:04:50.538000,NaT,1,1.0,None,,False,True
4,0 days 01:08:04.358000,VER,1,0 days 00:01:37.173000,5.0,1.0,NaT,NaT,0 days 00:00:31.255000,0 days 00:00:42.056000,...,False,Red Bull Racing,0 days 01:06:27.185000,NaT,1,1.0,None,,False,True


In [ ]:
# ── Cell 3: Jolpica API Query (Ergast successor) ─────────────────────────────

import json
from typing import Any, Dict, List
import requests

def fetch_driver_standings():

    # Use json format to avoid compatibility issues in the import
    endpoint = "https://api.jolpi.ca/ergast/f1/2024/driverStandings.json"
    timeout_seconds = 30

    try:
        # 1) Request data with explicit timeout.
        response = requests.get(endpoint, timeout=timeout_seconds)
        print(f"HTTP status code: {response.status_code}")

        # Requirement: status code must be 200.
        if response.status_code != 200:
            print(f"Warning: Unexpected HTTP status {response.status_code}")
            return # Salida limpia (Fail-Fast) sin romper el notebook

        # 2) Parse JSON payload.
        payload: Dict[str, Any] = response.json()

        # 3) Validate expected schema.
        mr_data = payload.get("MRData", {})
        standings_table = mr_data.get("StandingsTable", {})
        standings_lists: List[Dict[str, Any]] = standings_table.get("StandingsLists", [])

        if not standings_lists:
            print("Warning: No standings data found in API response.")
            return # Salida limpia (Fail-Fast)

        record_set = standings_lists[0].get("DriverStandings", [])
        print(f"Number of records returned: {len(record_set)}")
        print("-" * 40)



        print('\n── Data Structure Comparison ──')
        # Extract the columns of first register
        api_columns = list(record_set[0].keys())
        print(f"Jolpica 'Columns' (JSON keys): {api_columns}")


        # Print the structure of the first register of the json file
        print(json.dumps(record_set[0], indent=1))


        # 4) Print first 3 driver entries.
         # We prevent if there is no Given name to print the first 3 drivers
        print("First 3 driver entries (name, nationality, points):")
        for idx, entry in enumerate(record_set[:3], start=1):
            driver = entry.get("Driver", {})
            full_name = f"{driver.get('givenName', 'N/A')} {driver.get('familyName', 'N/A')}".strip()
            nationality = driver.get("nationality", "N/A")
            points = entry.get("points", "N/A")
            print(f"{idx}. {full_name} | {nationality} | {points} pts")
            
    except requests.exceptions.RequestException as e:
        print(f"Warning: API request failed. {e}")

# Ejecutamos la función
fetch_driver_standings()

HTTP status code: 200
Number of records returned: 24
----------------------------------------

── Data Structure Comparison ──
Jolpica 'Columns' (JSON keys): ['position', 'positionText', 'points', 'wins', 'Driver', 'Constructors']
{
 "position": "1",
 "positionText": "1",
 "points": "437",
 "wins": "9",
 "Driver": {
  "driverId": "max_verstappen",
  "permanentNumber": "3",
  "code": "VER",
  "url": "http://en.wikipedia.org/wiki/Max_Verstappen",
  "givenName": "Max",
  "familyName": "Verstappen",
  "dateOfBirth": "1997-09-30",
  "nationality": "Dutch"
 },
 "Constructors": [
  {
   "constructorId": "red_bull",
   "url": "https://en.wikipedia.org/wiki/Red_Bull_Racing",
   "name": "Red Bull",
   "nationality": "Austrian"
  }
 ]
}
First 3 driver entries (name, nationality, points):
1. Max Verstappen | Dutch | 437 pts
2. Lando Norris | British | 374 pts
3. Charles Leclerc | Monegasque | 356 pts


Data Shape 

1) a) I choose the FastF1 Bahrain race, this because it was an example of race, but more important his shape of the career it's interesting, allowing to analysis in the future some data about the time



2. b) The shape of both datasets it's different, in Jolpica it has 6 columns approximetely with 24 records in the sample, while in FastF1 it has 22 columns with 20 records in the sample.

3. c) 
It's interesting that there is more data in Fastf1, in Reality in Jolpica the 'Constructors' is a nested file with a more little more data, in the example it's about 16 columns, it's interesting to analize the data of FastF1. Will be interesting to compare the 2 datasets, what is the relation between two or hwo to join this 2 source of data


